# 04 — Analysis and Report Plots

Generate publication-quality figures and summary statistics for the LaTeX report.

In [ ]:
import sys, os
if os.path.basename(os.getcwd()) == 'notebooks':
    sys.path.insert(0, os.path.dirname(os.getcwd()))
else:
    sys.path.insert(0, os.getcwd())

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from src.utils import load_results, FIGURES_DIR, RESULTS_DIR

plt.rcParams.update({'font.size': 11, 'axes.labelsize': 12, 'figure.dpi': 150})

## 1. Load Results

In [ ]:
results = {}
for f in RESULTS_DIR.glob('*.json'):
    results[f.stem] = load_results(f.name)
    print(f'Loaded: {f.stem}')
print(f'\nAvailable: {list(results.keys())}')

## 2. Main Figure: TypiClust vs Random

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for key, c, mk, label, ls in [
    ('typiclust_euclidean_b10', 'tab:blue', 'o', 'TypiClust', '-'),
    ('random_b10', 'tab:gray', 's', 'Random', '--'),
]:
    if key not in results: continue
    r = results[key]; b = r['cumulative_budget']
    m = np.array(r['mean_accuracy']); s = np.array(r['se_accuracy'])
    ax.plot(b, m, f'{mk}{ls}', label=label, color=c, markersize=6)
    ax.fill_between(b, m-s, m+s, alpha=0.2, color=c)
ax.set_xlabel('Cumulative Budget'); ax.set_ylabel('Test Accuracy (%)')
ax.set_title('CIFAR-10: TypiClust vs Random'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'fig1_main.png'), dpi=300, bbox_inches='tight'); plt.show()

## 3. Modification Figure: Typicality Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
styles = {'euclidean': ('tab:blue','o'), 'cosine': ('tab:orange','^'), 'lof': ('tab:green','s'), 'kde': ('tab:red','D')}
for key, r in results.items():
    if 'typiclust_' in key:
        t = key.replace('typiclust_','').replace('_b10','')
        if t in styles:
            c, mk = styles[t]; b = r['cumulative_budget']
            m = np.array(r['mean_accuracy']); s = np.array(r['se_accuracy'])
            ax.plot(b, m, f'{mk}-', label=f'TypiClust ({t})', color=c, markersize=6)
            ax.fill_between(b, m-s, m+s, alpha=0.15, color=c)
if 'random_b10' in results:
    r = results['random_b10']
    ax.plot(r['cumulative_budget'], r['mean_accuracy'], 'x--', label='Random', color='tab:gray')
ax.set_xlabel('Cumulative Budget'); ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Typicality Measure Comparison'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'fig2_modification.png'), dpi=300, bbox_inches='tight'); plt.show()

## 4. Summary Table (LaTeX-ready)

In [ ]:
rows = []
for key, r in sorted(results.items()):
    b = r['cumulative_budget']; m = r['mean_accuracy']; s = r['se_accuracy']
    for i in range(len(b)):
        rows.append({'Method': key, 'Budget': b[i], 'Accuracy': f'{m[i]:.1f} $\\pm$ {s[i]:.1f}'})
df = pd.DataFrame(rows)
pivot = df.pivot(index='Budget', columns='Method', values='Accuracy')
print(pivot.to_string())
print('\nLaTeX table (copy-paste):')
print(pivot.to_latex(escape=False))

## 5. Key Statistics

In [ ]:
print('KEY STATS FOR REPORT')
print('='*50)
for k, r in sorted(results.items()):
    m = r['mean_accuracy']; s = r['se_accuracy']; b = r['cumulative_budget']
    print(f'{k}: B={b[-1]} -> {m[-1]:.1f}% +/- {s[-1]:.1f}%')

if 'typiclust_euclidean_b10' in results and 'random_b10' in results:
    tc = np.array(results['typiclust_euclidean_b10']['mean_accuracy'])
    rd = np.array(results['random_b10']['mean_accuracy'])
    print(f'\nTypiClust improvement: {(tc-rd).mean():+.1f}% avg across rounds')